# Colab 기본 GPU 런타임 검증
작성자 kimjaehak · 준비일 2026-11-23 · 상태 완료 · 실행·출력 검토일 2026-11-24 · T03

새 Colab GPU 런타임에서 위부터 실행하고 출력 포함 저장함. 합성 JPEG와 마스크로 환경을 점검함. 작은 배치의 가중치 갱신은 동작 확인용이며 테스트 데이터와 성능 측정은 사용하지 않음. Drive의 공통 스크립트와 공식 CAT-Net 가중치, torchvision 가중치 다운로드 및 공식 코드 복제를 위한 네트워크 접근이 필요함.

[검증 보고서](../../docs/eda/EXP_kimjaehak_007_colab_environment_check.md) · [U-Net 기준](../../docs/rgb_unet/EXP_kimjihwan_001_unet_pipeline_check.md) · [CAT-Net 기준](../../docs/catnet/EXP_kimjaehak_005_catnet_transfer_learning.md)


## 1. 기본 환경 기록과 CAT-Net 추가 패키지 설치
설치 전후 공통 패키지 버전을 대조함. 충돌 시 오류 출력을 보존함.


In [1]:
import importlib.metadata as metadata
import os
import platform
import subprocess
import sys
import tempfile
from datetime import datetime, timezone
from pathlib import Path

core = ["torch", "torchvision", "numpy", "Pillow", "pandas", "matplotlib"]
baseline = {pkg: metadata.version(pkg) for pkg in core}
print(datetime.now(timezone.utc).isoformat(), "Python", platform.python_version(), "OS", platform.platform())
print("Colab", metadata.version("google-colab"), "런타임 태그", os.getenv("COLAB_RELEASE_TAG", "미제공"))
print("설치 전", baseline)
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv"], check=True)

extras = ["pyjpegio==0.3.0", "yacs==0.1.8", "torch-dct==0.1.6", "PyYAML==6.0.3",
          "opencv-python==5.0.0.93", "tqdm==4.67.3"]
with tempfile.TemporaryDirectory() as tmp:
    constraints = Path(tmp) / "colab-defaults.txt"
    constraints.write_text("".join(f"{pkg}=={version}\n" for pkg, version in baseline.items()))
    subprocess.run([sys.executable, "-m", "pip", "install", "-c", str(constraints), *extras], check=True)
installed = {pkg: metadata.version(pkg) for pkg in core}
print("설치 후", installed)
assert installed == baseline, "기본 패키지의 설치 전후 버전을 확인해야 함."
print("CAT-Net 추가 패키지", {spec.split("==")[0]: metadata.version(spec.split("==")[0]) for spec in extras})


2026-11-24T02:05:18.253537+00:00 Python 3.13.16 OS Linux-6.6.159+-x86_64-with-glibc2.39
Colab 1.0.0 런타임 태그 release-colab-external-images_20261118-060058_RC00
설치 전 {'torch': '2.14.1+cu130', 'torchvision': '0.29.1+cu130', 'numpy': '2.4.6', 'Pillow': '11.3.0', 'pandas': '2.2.3', 'matplotlib': '3.10.0'}
설치 후 {'torch': '2.14.1+cu130', 'torchvision': '0.29.1+cu130', 'numpy': '2.4.6', 'Pillow': '11.3.0', 'pandas': '2.2.3', 'matplotlib': '3.10.0'}
CAT-Net 추가 패키지 {'pyjpegio': '0.3.0', 'yacs': '0.1.8', 'torch-dct': '0.1.6', 'PyYAML': '6.0.3', 'opencv-python': '5.0.0.93', 'tqdm': '4.67.3'}


## 2. Drive 코드·가중치 경로와 GPU 확인


In [2]:
import hashlib
import importlib
import numpy as np
import torch
from google.colab import drive
from PIL import Image
from torch import nn
from torch.nn import functional as F
from torchvision.models import ResNet50_Weights, resnet50

drive.mount("/content/drive")
project = Path("/content/drive/MyDrive/AI_Forensics/project")
weight_source = project / "models/pretrained/catnet/CAT_full_v2.pth.tar"
assert weight_source.is_file(), "Drive의 공식 CAT-Net 가중치를 확인해야 함."
sys.path.insert(0, str(project))
expected = {
    "__init__.py": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",
    "datasets.py": "435215ef08c2c43ce2f84795d783895074eee85f8e062f28ea8ee6acabad3e88",
    "catnet.py": "04d2d6561caf27ae83ccff844adf2b71c02d48470a37359d268f2ee8e217c889",
    "preprocessing.py": "73ce7fa0aee3f2b901a9ab04d430ff8ae2bcaa7262005600f6b08f04b95c652f",
    "evaluation.py": "ad400aa2cf453bd54ecc3e73abc38bc80cb1f7239eea7489b3072dae96dfcb52"
}
for filename, digest in expected.items():
    file = project / "scripts" / filename
    actual = hashlib.sha256(file.read_bytes()).hexdigest()
    print(filename, actual)
    assert actual == digest, "Drive 공통 코드 버전을 확인해야 함."
    module = "scripts" if filename == "__init__.py" else "scripts." + file.stem
    assert Path(importlib.import_module(module).__file__).resolve() == file.resolve()
from scripts import catnet, evaluation, preprocessing

assert torch.cuda.is_available(), "Colab GPU 런타임에 연결해야 함."
device = torch.device("cuda")
torch.manual_seed(42)
print("공통 코드 기준", "de0bb27607f5821c94f9cb21c2ea9854da181530")
print(torch.cuda.get_device_properties(device), "CUDA", torch.version.cuda, "cuDNN", torch.backends.cudnn.version())


Mounted at /content/drive
__init__.py e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855
datasets.py 435215ef08c2c43ce2f84795d783895074eee85f8e062f28ea8ee6acabad3e88
catnet.py 04d2d6561caf27ae83ccff844adf2b71c02d48470a37359d268f2ee8e217c889
preprocessing.py 73ce7fa0aee3f2b901a9ab04d430ff8ae2bcaa7262005600f6b08f04b95c652f
evaluation.py ad400aa2cf453bd54ecc3e73abc38bc80cb1f7239eea7489b3072dae96dfcb52
공통 코드 기준 de0bb27607f5821c94f9cb21c2ea9854da181530
_CudaDeviceProperties(name='Tesla T4', major=7, minor=5, total_memory=14912MB, multi_processor_count=40, uuid=47e5d942-9a60-fd83-07eb-a3f042f8546d, pci_bus_id=0, pci_device_id=4, pci_domain_id=0, L2_cache_size=4MB) CUDA 13.0 cuDNN 92800


## 3. 합성 JPEG·마스크 준비와 세 U-Net 검증
기준 U-Net의 가중치 초기화·디코더를 사용함. 동일한 합성 표본에서 RGB, ELA, RGB+ELA 입력을 만들고 각각 한 번 갱신한 뒤 임시 체크포인트를 새 모델에 로드해 예측을 대조함.


In [3]:
work = Path(tempfile.mkdtemp(prefix="t03_", dir="/content"))
jpeg = work / "sample.jpg"
rng = np.random.default_rng(42)
image = rng.integers(0, 180, size=(256, 256, 3), dtype=np.uint8)
mask = np.zeros(image.shape[:2], dtype=np.uint8)
mask[64:160, 96:192] = 1
image[mask == 1] = [220, 60, 40]
Image.fromarray(image).save(jpeg, quality=100, subsampling=0)
rgb = preprocessing.load_rgb(jpeg)
ela = preprocessing.make_ela(rgb, quality=90, scale=15, subsampling=2)
target = torch.from_numpy(mask).to(device=device, dtype=torch.float32)[None, None]
imagenet_mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
imagenet_std = np.array([0.229, 0.224, 0.225], dtype=np.float32)
input_channels = {"rgb": 3, "ela": 3, "rgb_ela": 6}


def to_input(rgb, ela, mode):
    parts = {"rgb": [rgb], "ela": [ela], "rgb_ela": [rgb, ela]}[mode]
    normalized = [(part.astype(np.float32) / 255.0 - imagenet_mean) / imagenet_std for part in parts]
    return torch.from_numpy(np.concatenate(normalized, axis=2).transpose(2, 0, 1).copy())


In [4]:
class DecoderBlock(nn.Module):
    def __init__(self, in_channels, skip_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels + skip_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels), nn.ReLU(inplace=True),
        )

    def forward(self, x, skip=None):
        x = F.interpolate(x, scale_factor=2, mode="nearest")
        if skip is not None:
            x = torch.cat([x, skip], dim=1)
        return self.block(x)


class ResNet50UNet(nn.Module):
    def __init__(self, in_channels):
        super().__init__()
        encoder = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
        if in_channels == 6:
            rgb_weight = encoder.conv1.weight.detach().clone()
            encoder.conv1 = nn.Conv2d(6, 64, 7, stride=2, padding=3, bias=False)
            with torch.no_grad():
                encoder.conv1.weight.copy_(torch.cat([rgb_weight, rgb_weight], dim=1))
        elif in_channels != 3:
            raise ValueError("입력 채널은 3 또는 6이어야 함.")
        self.stem = nn.Sequential(encoder.conv1, encoder.bn1, encoder.relu)
        self.pool = encoder.maxpool
        self.layers = nn.ModuleList([encoder.layer1, encoder.layer2, encoder.layer3, encoder.layer4])
        self.decoders = nn.ModuleList([
            DecoderBlock(2048, 1024, 256), DecoderBlock(256, 512, 128), DecoderBlock(128, 256, 64),
            DecoderBlock(64, 64, 32), DecoderBlock(32, 0, 16),
        ])
        self.head = nn.Conv2d(16, 1, 1)

    def forward(self, x):
        if x.shape[-2] % 32 or x.shape[-1] % 32:
            raise ValueError("입력 높이·너비는 32의 배수여야 함.")
        stem = self.stem(x)
        features = []
        out = self.pool(stem)
        for layer in self.layers:
            out = layer(out)
            features.append(out)
        out = features[-1]
        for decoder, skip in zip(self.decoders, [features[2], features[1], features[0], stem, None]):
            out = decoder(out, skip)
        return self.head(out)


In [5]:
def check_unet(mode, channels):
    model = ResNet50UNet(channels).to(device)
    x = to_input(rgb, ela, mode)[None].to(device)
    optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)
    initial = model.head.weight.detach().clone()
    logits = model(x)
    assert logits.shape == target.shape and logits.dtype == torch.float32
    loss = F.binary_cross_entropy_with_logits(logits, target)
    assert torch.isfinite(loss)
    loss.backward()
    assert torch.isfinite(model.head.weight.grad).all()
    optimizer.step()
    assert not torch.equal(initial, model.head.weight)
    model.eval()
    with torch.no_grad():
        before = model(x)
    checkpoint = work / f"{mode}.pt"
    torch.save(model.state_dict(), checkpoint)
    restored = ResNet50UNet(channels).to(device).eval()
    restored.load_state_dict(torch.load(checkpoint, map_location="cpu", weights_only=True))
    with torch.no_grad():
        after = restored(x)
    assert torch.isfinite(before).all() and torch.isfinite(after).all()
    torch.testing.assert_close(after, before)
    predicted = (after.sigmoid()[0, 0] >= 0.5).cpu().numpy()
    counts = evaluation.confusion_counts(mask, predicted)
    assert sum(counts) == mask.size
    assert evaluation.pixel_f1(*counts[:3]) is not None and evaluation.iou(*counts[:3]) is not None
    print(mode, "입력", tuple(x.shape), "loss", loss.item(), "재로드 최대 오차", (after - before).abs().max().item(), "평가 연결 통과")

for mode, channels in input_channels.items():
    check_unet(mode, channels)
torch.cuda.empty_cache()


Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 123MB/s] 


rgb 입력 (1, 3, 256, 256) loss 0.7591487169265747 재로드 최대 오차 0.0 평가 연결 통과
ela 입력 (1, 3, 256, 256) loss 0.8745319247245789 재로드 최대 오차 0.0 평가 연결 통과
rgb_ela 입력 (1, 6, 256, 256) loss 0.6069298386573792 재로드 최대 오차 0.0 평가 연결 통과


## 4. CAT-Net 공식 JPEG 입력·추론·출력부 갱신
005와 같이 출력부만 갱신함. 임시 점검값으로 학습한 결과는 환경 검증에만 사용함.


In [6]:
code_dir = catnet.prepare_code(work / "CAT-Net")
weight = catnet.prepare_weights(weight_source, work / weight_source.name)
detector = catnet.CatNetDetector.from_pretrained(code_dir, weight, device=device)
print(detector.info)
probability = detector.predict_proba(jpeg)
assert probability.shape == mask.shape and np.isfinite(probability).all()
tensor, qtable, original_size = catnet.prepare_input(jpeg, code_dir)
print("JPEG", original_size, "입력", tuple(tensor.shape), "양자화 테이블", tuple(qtable.shape))
model = detector.model
model.requires_grad_(False)
model.last_layer.requires_grad_(True)
optimizer = torch.optim.SGD(model.last_layer.parameters(), lr=1e-3)
initial = [p.detach().clone() for p in model.last_layer.parameters()]
logits = model(tensor[None].to(device), qtable[None].to(device))
assert logits.shape[:2] == (1, 2)
logits = F.interpolate(logits, size=original_size, mode="bilinear", align_corners=False)
loss = F.cross_entropy(logits, target[:, 0].long())
assert torch.isfinite(loss)
loss.backward()
assert all(p.grad is not None and torch.isfinite(p.grad).all() for p in model.last_layer.parameters())
optimizer.step()
assert all(torch.isfinite(p).all() for p in model.last_layer.parameters())
assert any(not torch.equal(p, old) for p, old in zip(model.last_layer.parameters(), initial))
print("CAT-Net JPEG 추론·출력부 1-step 갱신 통과", "loss", loss.item())


공식 코드: /content/t03_gnro1exv/CAT-Net
코드 커밋: 5eb85900b0e72423c95a8d41bc835b86c8671f8a
가중치 복사·해시 확인 완료: /content/t03_gnro1exv/CAT_full_v2.pth.tar
{'code_commit': '5eb85900b0e72423c95a8d41bc835b86c8671f8a', 'code_dir': '/content/t03_gnro1exv/CAT-Net', 'weight_path': '/content/t03_gnro1exv/CAT_full_v2.pth.tar', 'weight_sha256': 'f82aaafdd1142775231feedcea0bb7027f7370561d9e8d107465454001865989', 'weight_bytes': 915503873, 'epoch': 196, 'parameters': 114263810, 'device': 'cuda', 'wrapper_sha256': '04d2d6561caf27ae83ccff844adf2b71c02d48470a37359d268f2ee8e217c889', 'resize': 'bilinear', 'align_corners': False}


/content/t03_gnro1exv/CAT-Net/Splicing/data/AbstractDataset.py:163: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /__w/pytorch/pytorch/torch/csrc/utils/tensor_new.cpp:252.)
  t_DCT_coef = torch.tensor(DCT_coef, dtype=torch.float)  # final (but used below)
/content/t03_gnro1exv/CAT-Net/lib/models/network_CAT.py:543: UserWarning: `nn.functional.upsample` is deprecated. Use `nn.functional.interpolate` instead.
  x1 = F.upsample(x[1], size=(x0_h, x0_w), mode='bilinear')
/content/t03_gnro1exv/CAT-Net/lib/models/network_CAT.py:544: UserWarning: `nn.functional.upsample` is deprecated. Use `nn.functional.interpolate` instead.
  x2 = F.upsample(x[2], size=(x0_h, x0_w), mode='bilinear')
/content/t03_gnro1exv/CAT-Net/lib/models/network_CAT.py:545: UserWarning: `nn.functional.upsample` is deprecated. Use `nn.functional.interpolat

JPEG (256, 256) 입력 (24, 256, 256) 양자화 테이블 (1, 8, 8)
CAT-Net JPEG 추론·출력부 1-step 갱신 통과 loss 0.03116569295525551


모든 셀 실행 후 표준 출력과 오류를 포함해 로컬 노트북을 저장함. 실패하면 실패한 셀의 출력과 배정 GPU를 함께 보존함.
